
###Display all Scope.  

In [0]:
dbutils.secrets.listScopes()


###Display all secrects for particular Scope

In [0]:
dbutils.secrets.list(scope='Banking-Scope-project01')

###Create mount point for Bronze container

In [0]:
spark.conf.set(
    "fs.azure.account.auth.type.bankingstorageprojecto1.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.bankingstorageprojecto1.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.bankingstorageprojecto1.dfs.core.windows.net",
    dbutils.secrets.get(scope="Banking-Scope-project01", key="client-id")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.bankingstorageprojecto1.dfs.core.windows.net",
    dbutils.secrets.get(scope="Banking-Scope-project01", key="client-Secrets")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.bankingstorageprojecto1.dfs.core.windows.net",
    "https://login.microsoftonline.com/65583477-d432-469c-b4af-ab046853916f/oauth2/v2.0/token",
)



###Display all files bronze container 

In [0]:
dbutils.fs.ls("abfss://bronze@bankingstorageprojecto1.dfs.core.windows.net/")

###Create widget for catalog, schema and Table

In [0]:
dbutils.widgets.text("Catalog_Name","")
dbutils.widgets.text("Schema_Name","")
dbutils.widgets.text("Table_Name","")

In [0]:
Catalog_Name = dbutils.widgets.get("Catalog_Name")
Schema_Name = dbutils.widgets.get("Schema_Name")
Table_Name = dbutils.widgets.get("Table_Name")


###Create autoloader for continuous loding data

In [0]:
from pyspark.sql.functions import input_file_name



bronze_table_path = f"{Catalog_Name}.{Schema_Name}.{Table_Name}"
checkpoint_path = f"/Volumes/{Catalog_Name}/{Schema_Name}/metadata_files/checkpoints/{Table_Name}"


container_name = "abfss://bronze@bankingstorageprojecto1.dfs.core.windows.net/" 

if(spark.catalog.tableExists('bronze_table_path')):
    includeExistingFiles_value="False"
else:
    includeExistingFiles_value="True"


# Read data from adlsgen2 container using AutoLoader
df = (
    spark.readStream.format("cloudFiles")
    .option(
        "cloudFiles.format", "csv"
    )  # Change format as needed (csv, json, parquet, etc.)
    .option(
        "cloudFiles.inferSchema", "true"
    )  # Set to "false" if using a predefined schema
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.includeExistingFiles", includeExistingFiles_value)
    .load(container_name)
    .withColumn("source_file", input_file_name())
)

# Write data to Bronze Delta Table in append mode
(
    df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(once=True)
    .toTable(f'{bronze_table_path}')
    .awaitTermination()
    
)